In [ ]:
import json
from pathlib import Path

import torch
import h5py
import numpy as np
from ipywidgets import interact
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
# Batch size and dataloaders
from torch.utils.data import Dataset, DataLoader
# TT split
from sklearn.model_selection import train_test_split
# Training

In [ ]:
VAL_TEST_SPLIT = (0.083, 0.083)
ID = "cylinder_Re41_interp"
NAME = "Cylinder (steady, laminar, Re=41)"
RANDOM_STATE = 42

BATCH_SIZE = 8

In [ ]:
DATASET_INPUT = "cylinder_8000_interp"
dataset_path = Path.cwd().parents[1] / f"FluidLearn/data/02_sampled/{DATASET_INPUT}"

In [ ]:
### Load data
cases = sorted([file.name for file in dataset_path.iterdir() if file.suffix == ".h5" and file.stem != "constants"], key=lambda x: float(x.split(".")[0]))
N_cases = len(cases)
# chosen_cases = np.linspace(0, N_cases-1, 20).astype(int) # Use 20 evenly spaced cases
# chosen_cases = np.repeat(N_cases-1, 20).astype(int)
chosen_cases = np.repeat(0, 1).astype(int)
# chosen_cases = np.arange(0, N_cases, 1).astype(int) # Use all cases
data_bycase = [] # Structure: case_name -> snapshot -> input / output -> input scalars / input fields // target fields
keys = ["p", "Ux", "Uy"]

with h5py.File(dataset_path / "constants.h5", "r") as file:
    validity = file['validity'][:].astype(np.float32)
    sdf = file['sdf'][:].astype(np.float32)
sdf = sdf.clip(-sdf.max(), None)

for case_name in chosen_cases:#range(len(cases))
    case_path = dataset_path / cases[case_name]
    
    with h5py.File(case_path, "r") as file:
        data = {field: file[field][:].astype(np.float32) for field in keys}
        fields = np.stack([data[key] for key in keys], axis=1)  # Shape: (snapshots, fields, x, y)
        t = file['t'][:].astype(np.float32)
        
        # Access attributes
        nu = file.attrs['nu'].astype(np.float32)
        Re = 2.0 / nu

    fields[np.isnan(fields)] = 0.0

    t = t[1:]
    fields = fields[1:]
    # fields = np.concatenate([fields, np.tile(sdf, (fields.shape[0], 1, 1, 1))], axis=1)

    Nnaps, Nfields = fields.shape[:2]

    dt = t[1:] - t[:-1]

    data_bycase.append([
        np.concatenate([np.tile([Re], (Nnaps-1, 1)), dt.reshape(-1, 1)], axis=1),  # Input scalars
        fields[:Nnaps-1],    # Input fields
        fields[1:Nnaps]#-fields[:Nnaps-1],       # Target fields
    ])

# # Separate train/validation from test cases
# test_cases = [len(data_bycase) // 2]
# train_cases = [i for i in range(len(data_bycase)) if i not in test_cases]
# data_bycase_train = [data_bycase[i] for i in train_cases]

data_bycase_train = data_bycase
data_raw = [
    np.concatenate([case_name[0] for case_name in data_bycase_train]),  # Input scalars
    np.concatenate([case_name[1] for case_name in data_bycase_train]),  # Input fields
    np.concatenate([case_name[2] for case_name in data_bycase_train]),  # Target fields
]

width = data_raw[1].shape[2]
height = data_raw[1].shape[3]
print(data_bycase[0][0].shape, data_bycase[0][1].shape, data_bycase[0][2].shape)

In [ ]:
scalars, fields, targets = data_raw

### TT Split
train_idx, test_idx = train_test_split(np.arange(len(scalars)), test_size=VAL_TEST_SPLIT[1], random_state=RANDOM_STATE)
train_idx, val_idx = train_test_split(train_idx, test_size=VAL_TEST_SPLIT[0] / (1 - VAL_TEST_SPLIT[1]), random_state=RANDOM_STATE)

In [ ]:
output_dir = Path.cwd() / ID
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "id": ID,
    "name": NAME,
    "trainingType": "synthetic",
    "validationType": "synthetic",
    "compressibility": "Incompressible",
    "image": "/assets/cylinder-flow.png",
    "description": "A canonical wake prediction task: use a sequence of velocity and pressure snapshots to predict the future incompressible flow around a cylinder.",
    "mesh": "Structured",
    "reynoldsNumbers": [41],
    "inputScalarNames": ["Re", "dt"],
    "flowVariables": keys,
    "splitFractions": {"train": 1 - sum(VAL_TEST_SPLIT), "validation": VAL_TEST_SPLIT[0], "test": VAL_TEST_SPLIT[1]},
    "sample_count": len(targets),
    "width": width,
    "height": height,
}
split_indices = (("train", train_idx), ("test", test_idx), ("validation", val_idx))
for split_name, indices in split_indices:
    output_path = output_dir / f"{split_name}.h5"
    with h5py.File(output_path, "w") as file:
        inputs = file.create_group("inputs")
        inputs.create_dataset("scalars", data=scalars[indices], compression="gzip", shuffle=True)
        inputs.create_dataset("fields", data=fields[indices], compression="gzip", shuffle=True)
        targets_group = file.create_group("targets")
        targets_group.create_dataset("fields", data=targets[indices], compression="gzip", shuffle=True)

    print(f"Saved {split_name} dataset to {output_path.resolve()}")

metadata_path = output_dir / f"{ID}.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved unified metadata to {metadata_path.resolve()}")